# Vegetation Index Crop Insurance (VICI)

VICI is an NDVI-based drought monitoring service designed for micro-insurance purposes.

This notebook guides you through all the steps required to prepare and run the VICI workflow.

The VICI product has been designed by prof. Kees de Bie at (ITC, University of Twente). VITO has converted this workflow into an operational service on Terrascope's EOPlaza platform for Ethiopia. 
More information: https://portal.terrascope.be/catalogue/app-details/186

First some preparations:

In [ ]:
!pip install ipykernel ipyleaflet geopandas geojson loguru --quiet

In [ ]:
# Set output directory for your run
from pathlib import Path
from datetime import datetime

now = datetime.now().strftime('%Y%m%d_%H%M%S')
outdir = Path(f'./results/vici/run_{now}')
outdir.mkdir(parents=True, exist_ok=True)

### Draw your region of interest

In [ ]:
from vito_agri_tutorials.utils.map import ui_map

map = ui_map()

In [ ]:
# Get the AOI from the map and save as GeoPackage file
aoi_gdf = map.get_objects()

# Save to geopackage file
aoi_gpkg_file = outdir / "AOI.gpkg"
aoi_gdf.to_file(aoi_gpkg_file, driver="GPKG")

### Preparations: process 20 year NDVI archive

#### Step 1: Define archive period

In [ ]:
from vito_agri_tutorials.vici import get_vici_archive_dates

# Specify final year of 20-year reference period
end_year_archive = 2019

start_date_archive, end_date_archive = get_vici_archive_dates(end_year_archive)

#### Step 2: Gather required NDVI imagery

The `get_ndvi_data_terrascope` function:
- Adds 6 dekads before and after the 20 years time period (for data cleaning step - see step 3)
- Clips your AOI from the global CGLOPS NDVI collection available on Terrascope (/data/MTDA/Copernicus/Land/global/netcdf/ndvi/)
- Converts 300 m data to 1 km data
- Deletes data for dekads without any valid observation (based on NOBS data as produced by data provider) - these dekads are automatically set to 0

The original NDVI data are stored as individual .tif files per dekad in the NDVI_archive/NDVI_original subdirectory.

In [ ]:
from vito_agri_tutorials.vici import get_ndvi_data_terrascope

archive_dir = outdir / "NDVI_archive"
ndvi_archive_dir = archive_dir / "NDVI_original"
get_ndvi_data_terrascope(
        aoi_gpkg_file, ndvi_archive_dir, start_date_archive, end_date_archive
    )

Let's inspect an individual NDVI image!

(Note that the data is originally stored as DN (digital numbers between 0 and 250) and need to be converted to physical NDVI values using the default scale and offset parameters)

In [ ]:
import glob
from matplotlib import pyplot as plt
from vito_agri_tutorials.utils.geotiff import read_geotiff
from vito_agri_tutorials.vici import NDVI_SCALE, NDVI_OFFSET

# Get the files
infiles = sorted(glob.glob(str(ndvi_archive_dir / "*.tif")))
infile = infiles[0]
print('File to load:')
print(infile)

# Read the file
ndvi_data = read_geotiff(infile)
print('Range of loaded data:', ndvi_data.min(), ndvi_data.max())

# rescale the NDVI to meaningful values
ndvi_data = (ndvi_data * NDVI_SCALE) + NDVI_OFFSET
print('Range of rescaled data:', ndvi_data.min(), ndvi_data.max())

# Visualize the image
plt.imshow(ndvi_data, cmap='RdYlGn')
plt.colorbar(label='NDVI')
plt.title('Normalized Difference Vegetation Index (NDVI)')
plt.show()

Now let's see how a pixel time series looks like:

In [ ]:
import numpy as np

# Now load timeseries for one pixel and one year
ndvi_ori = []
infiles = sorted(glob.glob(str(ndvi_archive_dir / "*.tif")))
files_to_load = infiles[0:108]  # First 3 years (108 dekads)
for file_path in files_to_load:
    ndvi  = read_geotiff(file_path)
    ndvi_ori.append(ndvi)
ndvi_ori = np.array(ndvi_ori)

# Extract values for one pixel
pixel_ori = ndvi_ori[:, 5, 5]

# Convert to meaningful values
pixel_ori = (pixel_ori * NDVI_SCALE) + NDVI_OFFSET

# Plot the timeseries
fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(pixel_ori, '-b', label='Original NDVI')
ax.set_xlabel('Dekad number')
ax.set_ylabel('NDVI')
ax.legend()

#### Step 3: Apply upper envelope smoothing

Despite initial pre-processing by the Copernicus Global Land data provider, the NDVI data can still contain quite some noise. We apply an adapted Savitzky-Golay smoothing function to get rid of most of this noise.

Note that whenever a pixel contains a consecutive sequence of more than 12 dekads with no data, the entire pixel becomes invalid during this process!


In [ ]:
from vito_agri_tutorials.vici import upper_envelope_smoothing

ndvi_smoothed_file = archive_dir / "NDVI_smoothed.tif"
upper_envelope_smoothing(
        ndvi_archive_dir, start_date_archive, end_date_archive, ndvi_smoothed_file
    )

Now let's visualize the impact of the smoothing procedure:

In [ ]:
from vito_agri_tutorials.utils.geotiff import read_geotiff
from matplotlib import pyplot as plt
import glob
import numpy as np

# Get original NDVI data from separate files
ndvi_ori = []
infiles = sorted(glob.glob(str(ndvi_archive_dir / "*.tif")))
for file_path in infiles:
    ndvi  = read_geotiff(file_path)
    ndvi_ori.append(ndvi)
ndvi_ori = np.array(ndvi_ori)

# Extract values for one pixel
pixel_ori = ndvi_ori[:, 5, 5]
pixel_ori = (pixel_ori * NDVI_SCALE) + NDVI_OFFSET

# Get smoothed NDVI data
smoothed = read_geotiff(ndvi_smoothed_file)

# extract the same pixel
pixel_smoothed = smoothed[:, 5, 5]
pixel_smoothed = (pixel_smoothed * NDVI_SCALE) + NDVI_OFFSET

# Extend the smoothed pixel timeseries with 6 dekads before and 6 dekads after
pixel_smoothed = np.concatenate((np.array(np.repeat(np.nan, 6)), pixel_smoothed, np.array(np.repeat(np.nan, 6))))

fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(pixel_ori, '-b', label='Original NDVI')
ax.plot(pixel_smoothed, '-r', label='Smoothed NDVI')
ax.set_xlabel('Dekad number')
ax.set_ylabel('NDVI')
ax.legend()

#### Step 4: Define invalid pixels

Now we determine for each pixel, based on the available NDVI data (20 years), whether or not we will include that pixel in our analysis.

We apply two checks:
1. If no NDVI data is available after smoothing, the pixel is discarded
2. If there is a clear trend present over the 20 years archive for the pixel, the pixel is discarded (slope parameter of trend should NOT be < -0.088983912 OR > +0.0755551367)

The following cell will output two files:
- invalid_pixel_mask.tif --> indicates invalid pixels with value 1
- invalid_pixel_mask_classified.tif --> specifies which of the above criteria led to discarding a certain pixel (1 = no data, 2 = trend)

In [ ]:
from vito_agri_tutorials.vici import create_invalid_pixel_mask

invalid_pixel_mask_file = archive_dir / "invalid_pixel_mask.tif"

invalid_pixel_mask = create_invalid_pixel_mask(
    ndvi_smoothed_file, invalid_pixel_mask_file
)

#### Step 5: Compute long-term statistics for each dekad to prepare for clustering

For each pixel and dekad combination we now have 20 NDVI values. Based on these values we compute:
- 10th percentile
- 50th percentile
- 90th percentile
- standard deviation

--> this results in an image of (4 metrics x 36 dekads =) 144 bands

NOTE: if a pixel contains more than 17/20 invalid NDVI observations for a dekad, the pixel/dekad combination are ignored


In [ ]:
from vito_agri_tutorials.vici import compute_stats_per_dekad

stats_per_dekad_file = archive_dir / "stats_per_dekad.tif"
stats_per_dekad = compute_stats_per_dekad(
        ndvi_smoothed_file, invalid_pixel_mask_file, stats_per_dekad_file
    )

#### Step 6: Define Crop Production Zones (CPSZs)

Based on the computed statistics per pixel, we run an ISODATA clustering approach to automatically define Crop Production Zones.<br>
ISODATA clustering automatically defines the optimal amount of clusters (within a provided limit).

As the ISODATA clustering algorithm is computationally intensive, we use a similar but faster k-means clustering approach in this notebook.

Based on the spatial extent selected by the user and the NDVI variability within, the user needs to set a minimum and maximum number of zones for the algorithm.<br>
The subsample parameter determines to what extent the data first needs to be reduced before applying the clustering algorithm (this will save time!).

In [ ]:
import matplotlib.pyplot as plt
from vito_agri_tutorials.utils.geotiff import read_geotiff
from vito_agri_tutorials.vici import determine_clusters_kmeans

cpsz_file = archive_dir / "cpsz.tif"

determine_clusters_kmeans(
        stats_per_dekad_file,
        cpsz_file,
        min_zones=2,
        max_zones=5,
        sub_sample=10, # set to 1 if you want to use all pixels
    )

# Visualize the clusters:
cpsz = read_geotiff(cpsz_file)
plt.imshow(cpsz, cmap='Spectral')
plt.colorbar(label='Zones')
plt.title('Crop production zones')
plt.show()

#### Step 7: Compute payout thresholds

Now that we have our final 20 years NDVI archive, we can compute the pay-out thresholds per zone:
- trigger = p15
- exit = p5

During this procedure, we also compute the median NDVI profile per zone, which will be used in a next step to define the growing seasons.

In [ ]:
from vito_agri_tutorials.vici import compute_payout_thresholds

final_thresholds_dir = archive_dir / "final_thresholds"

percentiles, p50_array = compute_payout_thresholds(
        ndvi_smoothed_file, cpsz_file, final_thresholds_dir
    )

# Summary plot showing all zones together
fig, ax = plt.subplots(figsize=(12, 6))
for z in range(p50_array.shape[1]):
    ax.plot(p50_array[:, z], label=f'Zone {z+1}', linewidth=2)

ax.set_title('Median NDVI Profiles by Zone')
ax.set_xlabel('Dekad')
ax.set_ylabel('NDVI')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Step 8: Derive growing period per zone

By default, the growing period is defined here as the period a crop is actually growing (senescence phase is not included as this phase is not important for drought monitoring).

In short, for each 9 dekad period we compute the average NDVI value. If the dekad value is larger than the this 9-dekad average, the dekad is considered "in-season".

In [ ]:
from vito_agri_tutorials.vici import define_growing_seasons

outdir_seasons = archive_dir / "seasons"

seasons = define_growing_seasons(p50_array, cpsz_file, outdir_seasons)
seasons


In [ ]:
# Visualize growing seasons
nzones = seasons.shape[1]
fig, axes = plt.subplots(nzones, 1, figsize=(12, 3*nzones))

# Handle case where there's only one zone (axes won't be a list)
if nzones == 1:
    axes = [axes]

for zone in range(nzones):
    # Primary axis for NDVI line plot
    ax1 = axes[zone]
    ax1.plot(p50_array[:, zone], '--', label='Median NDVI (p50)', color='blue', linewidth=2)
    ax1.set_xlabel('Dekad')
    ax1.set_ylabel('NDVI', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.set_title(f'Growing Season - Zone {zone+1}')
    
    # Secondary axis for growing season bars
    ax2 = ax1.twinx()
    
    # Create bars for growing seasons (1 = in season, 0 = out of season)
    season_bars = ax2.bar(
        range(36), 
        seasons[:, zone], 
        alpha=0.3, 
        color='green', 
        label='Growing Season'
    )
    
    # Configure secondary axis
    ax2.set_ylabel('Growing Season', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    ax2.set_ylim(0, 1.2)  # Set limits to make bars visible
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['Out of Season', 'In Season'])
    
    # Optional: Add grid
    ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Compute VICI for period of interest

We now have everything in place to start the actual monitoring phase in which we compare the current NDVI behavior against our 20 year archiving period, on a per-dekad basis.<br>
You now only need to provide your period of interest to run computations.<br>
For testing the service, we advise to run a maximum of one year at a time.

In [ ]:
from vito_agri_tutorials.vici import run_vici

# Define start and end date of interest
start_date = "2021-01-01"
end_date = "2021-12-21"

# Run the VICI processing
run_vici(outdir, start_date, end_date)

In the output folder you will now see two types of files per dekad:

- VICI_{dekad}.tif --> contains drought severity value [0 - 100] for this dekad

- quality_flags.tif --> contains information on why a certain pixel was not included and whether or not it is currently in season or not
    * 252: for this dekad, the pixel initially did not have NDVI data available. Due to the upper envelope smoothing, data has been added. Indicates that uncertainty for this pixel-dekad combination is likely a bit higher.
    * 253: the pixel is currently not in the growing period
    * 254: this pixel has been dropped due to not enough data or clear trend in 20 year archive
    * 255: this pixel has not been considered during the analysis for other reasons than mentioned in flag 254


In the cells below, we summarize the results we obtained using statistics and visuals:

In [ ]:
# Compute VICI statistics for each zone
from vito_agri_tutorials.vici import compute_vici_zonal_stats

vici_zonal_stats = compute_vici_zonal_stats(outdir)

vici_zonal_stats

# the results show the minimum, maximum, average VICI value per zone
# additionally, the freq metric shows the percentage of VICI values > 0 in a given over all available observations in a zone.

In [ ]:
# Visualize VICI raster and quality flags for one dekad
from vito_agri_tutorials.vici import show_dekadal_vici_result

dekad = '20211121'

show_dekadal_vici_result(outdir, dekad)

In [ ]:
# Visualize result for one pixel
from vito_agri_tutorials.vici import show_vici_result_pixel

pixel = (16, 16)

show_vici_result_pixel(outdir, pixel[0], pixel[1], start_date, end_date)

## NOTE on scaling up VICI computations:

In order to run this service in an operational way, there are a few adaptations to this workflow which are beyond the scope of this demo:

- switch to ISODATA clustering method to determine Crop Production Zones
- additional correction of NDVI data for each production zone separately

The VICI workflow can be run operationally at country scale through the Terrascope platform.
More information: https://portal.terrascope.be/catalogue/app-details/186

End of the exercise!